### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/hpo/DiT_Generation_HPO_Runner.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Ffiles%2Fnotebooks%2Fhpo%2FDiT_Generation_HPO_Runner.ipynb)

- **Colab:** click the button, choose **Runtime > Change runtime type > GPU**, and run the setup cell.
- **Kaggle:** click the button, sign in and import the notebook, then enable **Internet** and select a **GPU** in notebook settings before running setup.
- **Remote GPU container:** open this notebook inside the matching checkout with TensorFlow 2.20.0 / Keras 3.11.2 already installed. Setup verifies the environment without replacing packages.

Run the search cells individually to review the 100-trial stage before continuing.
Hosted setup uses the repository's shared installer for **TensorFlow 2.20.0 / Keras 3.11.2**;
restart the session and rerun setup if the installer requests it. CPU-only sessions
cannot run this HPO workflow.

These buttons open the **published GitHub `main` revision**. Publish this notebook,
its `common/dit_hpo_*.py` helpers, `common/gpu_resource_slots.py`, and the updated
shared `common/hpo*.py` APIs, `common/train.py`, and
`common/callbacks/hpo_pruning.py` together before using them. For an unpublished version, upload the notebook and the matching
complete checkout to the online runtime; setup reuses an existing checkout.

**Keep your results:** hosted disks can disappear when a session ends. Save the
complete `RESULTS_PATH` directory, including SQLite, sampler state, trial artifacts,
and `notebook_runner`. Restore it to the same absolute path with the same source,
Python and package versions to resume. Provider session limits still apply;
this notebook cannot keep an online runtime alive or remove its time limits.

# DiT generation HPO: search and seed confirmation

Optimize the standalone conditional **diffusion_transformer** on CIFAR-10 through
the codebase's existing HPO API. The scalar objective is post-training validation
**EMA noise_loss**, minimized. The classifier branch is absent.

Protocol: inspect the first **100** successful trials, reach **200**, optionally
add **50** to reach **250**, then rerun the top **three** configurations
on **three fresh paired seeds**. Set 300 explicitly for a larger extension.
The TPE sampler has a **40-observation random-startup threshold**; parallel
trials already allocated can extend the random-startup phase. Every search and confirmation run uses
the same **50-epoch maximum** and **early-stopping patience 5**. Search trials
also use conservative performance pruning; confirmations do not. Recognized
memory-exhaustion exceptions prune the affected search trial so other trials
can continue.

The notebook is delivered unexecuted. Search and confirmation cells launch real
training when you execute them. Run cells individually to review each stage.
No campaign deadline is imposed. Every allocated search trial consumes the
separate 400-attempt ceiling, including OOM pruning, performance pruning and
failures. Only finite COMPLETE trials satisfy the successful-trial target.

## GPU selection and parallel trials

Use a fresh GPU kernel on **Colab, Kaggle, or a remote Linux GPU container**.
The notebook kernel coordinates processes without importing TensorFlow or Keras.
Hosted setup uses the shared installer; a prepared container uses its existing
TensorFlow **2.20.0** / Keras **3.11.2** environment.

**GPU_IDS** selects physical GPU indices from `nvidia-smi`. **CONCURRENT_TRIALS**
is the total number of isolated trial workers across those devices:

| GPU_IDS | CONCURRENT_TRIALS | Search workers |
| --- | --- | --- |
| `[0]` | `1` | One trial on GPU 0 |
| `[0]` | `5` | Five parallel trials on GPU 0 |
| `[0, 1]` | `10` | Five parallel trials on each GPU |
| `[0, 1]` | `34` | 17 trials per GPU (tested default: two A100 80 GB GPUs) |

The default was capacity-tested on **two A100-SXM4-80GB GPUs** on 2026-10-08. The selected per-worker TensorFlow cap is **3,584 MiB**; the largest observed allocation in its validation probes was **3,370 MiB**. The checks exercised large plain and U-shaped DiTs, batch size 128, both CNN components, AdamW, EMA, full CIFAR-10 data loading, validation, TensorBoard, and final reporting. These are bounded capacity checks, not a completed 50-epoch HPO campaign. See the [measurement report](../../results/dit_hpo_concurrency_20261008_f888515c257b/REPORT.md) for exact workloads and the higher-setting boundary test.

Set both GPU selection and concurrency for your runtime before continuing; this default applies to the tested two-GPU container.

Select no more GPUs than workers. Each trial uses exactly one GPU; a single
coordinator calls **common.hpo.run_hpo** and owns the Optuna study. A finished
worker's slot is reused on the same GPU, so faster trials cannot overfill a device.
The setup binds selected devices to their measured UUIDs.

Parallel search waits for all requested reservations. Memory is accounted for
separately on each GPU; their VRAM is not pooled. The common per-worker TensorFlow
cap fits the smallest device budget after worker overhead and headroom. The
settings cell displays device assignments and budgets. Existing shared-pool
policies still apply, unknown ownership blocks admission, and existing jobs are
preserved. Memory admission does not guarantee that every sampled model fits its cap.

The CPU coordinator keeps GPU visibility disabled. Each training process is
registered before TensorFlow starts. Interrupting a search stops and reaps its
workers before releasing their reservations. Seed confirmations remain serial
on the first selected GPU, using the existing training and TensorBoard APIs.

In [ ]:
import os
import platform
import sys

from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"

# Reject laptop kernels before cloning a checkout or installing any packages.
if platform.system() != "Linux":
    raise RuntimeError("Use an online GPU runtime; laptop execution is prohibited.")
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    RUNTIME = "kaggle"
elif os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
    RUNTIME = "colab"
elif Path("/.dockerenv").is_file() or Path("/run/.containerenv").is_file():
    RUNTIME = "container"
else:
    raise RuntimeError("Use Colab, Kaggle, or a remote Linux GPU container.")

# GPU work happens only in admitted children, never in this coordinator kernel.
if "tensorflow" in sys.modules or "keras" in sys.modules:
    raise RuntimeError("Restart the kernel and run setup before importing TensorFlow or Keras.")
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["CONTINUAL_RUNTIME"] = RUNTIME
# Reuse the same standard-library initializer as the other online notebooks.
_locations = (
    Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
    Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME
)
_initializer = next((
    path / "files" / "notebooks" / "init.py" for path in _locations
    if (path / "files" / "notebooks" / "init.py").is_file()
), None)
_url = (
    f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/"
    f"{REVISION}/files/notebooks/init.py"
)
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
CHECKOUT_ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime="local" if RUNTIME == "container" else RUNTIME, 
    cuda=False, 
    install=False if RUNTIME == "container" else None
)

_REQUIRED_HELPERS = (
    "common/dit_hpo_runner.py", "common/dit_hpo_remote.py", 
    "common/dit_hpo_confirmation.py", "common/gpu_resource_slots.py"
)
if not all((CHECKOUT_ROOT / relative).is_file() for relative in _REQUIRED_HELPERS):
    raise RuntimeError(
        "The checkout is missing this runner's helpers. Publish or upload the matching "
        "complete checkout together with the notebook, then rerun setup."
    )
os.chdir(CHECKOUT_ROOT)
if str(CHECKOUT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_ROOT))
os.environ.setdefault("KERAS_HOME", str(CHECKOUT_ROOT.parent / ".keras"))

from common.dit_hpo_remote import inspect_remote
from common.dit_hpo_runner import (
    make_plan, search_summary, run_search, freeze_finalists, 
    run_confirmations, confirmation_summary
)


runtime = inspect_remote(CHECKOUT_ROOT)
display({key: value for key, value in runtime.items() if key not in {"source_sha256", "worker_policy", "concurrent_trials"}})
print("Fingerprinted source files:", len(runtime["source_sha256"]))

In [ ]:
DATASET = "CIFAR10"
RESULTS_PATH = "files/results/dit_generation_hpo_v6"
GPU_IDS = [0, 1]  # Use [0, 1] to select two GPUs.
CONCURRENT_TRIALS = 34  # Tested total across two A100 80 GB GPUs; 17 per GPU.
EPOCHS = 50
N_STARTUP_TRIALS = 40
SEARCH_SEED = 42

PRUNING = {
    "type": "percentile", 
    "monitor": "val_noise_loss", 
    "percentile": 75.0, 
    "n_startup_trials": 40, 
    "n_warmup_steps": 9,  # Zero-based epochs: first eligible check is epoch 10.
    "interval_steps": 5, 
    "n_min_trials": 10
}

REVIEW_TARGET = 100
SEARCH_TARGET = 200
EXTENSION_TARGET = 250  # Add 50 successful trials after the 200-trial main search.
MAX_ATTEMPTS = 400
BATCH_TRIALS = max(10, CONCURRENT_TRIALS)

TOP_K = 3
CONFIRMATION_SEEDS = [101, 202, 303]

plan = make_plan(
    CHECKOUT_ROOT, 
    RESULTS_PATH, 
    dataset_name=DATASET, 
    epochs=EPOCHS, 
    n_startup_trials=N_STARTUP_TRIALS, 
    concurrent_trials=CONCURRENT_TRIALS, 
    gpu_ids=GPU_IDS, 
    pruning=PRUNING, 
    seed=SEARCH_SEED
)
display(plan["hpo"])
print("Persistent study:", plan["study_root"])
WORKER_GPU_MEMORY_LIMIT_MB = plan["identity"]["worker_policy"]["tf_memory_mib"]
TENSORBOARD_DIRECTORY = Path(plan["study_root"])
print("Per-worker TensorFlow cap (MiB):", WORKER_GPU_MEMORY_LIMIT_MB)
display([{key: gpu[key] for key in ["gpu_id", "gpu_uuid", "gpu_name", "concurrent_trials", "worker_policy"]}
        for gpu in plan["identity"]["gpus"]])
print("TensorBoard root:", TENSORBOARD_DIRECTORY)

## Search-trial pruning and memory exhaustion

The shared HPO API uses Optuna's **PercentilePruner** on **val_noise_loss**,
the existing validation EMA noise MSE. A percentile of **75** retains the better
75% of reference performance, providing a conservative screen for poor settings.
A trial's best reported value so far is compared with completed trials' values
at the same epoch; a bad single epoch does not by itself define an outlier.

Pruning begins only after **40 completed reference trials**, no earlier than
**epoch 10**, and is checked every **5 epochs** thereafter. At least **10 completed
reference trials** must have reported that epoch. These are pruning safeguards;
they do not add trials to the 100 -> 200 -> 250 search targets. NaN/Inf loss guards
remain separate and stop numerical divergence without waiting for warmup.

Workers report epoch metrics to the single CPU coordinator. Only that coordinator
calls Optuna's report/prune APIs and writes the study database, including during
multi-GPU search. Pruned trials retain intermediate values, reason/evidence, and
TensorBoard metrics, release their GPU slot, and are recorded as **PRUNED**.
They count toward **MAX_ATTEMPTS=400**, but not toward the **250 successful trials**.
The target is therefore not guaranteed to fit within the attempt ceiling.

**Out-of-memory handling is separate from performance pruning.** A recognized
TensorFlow `ResourceExhaustedError` or Python `MemoryError` records the search
trial as **PRUNED** for resource exhaustion. This applies even with
**PRUNING = None** and does not wait for the startup or epoch warmup. The
original exception remains in trial metadata and worker logs, and TensorBoard
receives an OOM diagnostic. Any epoch metrics already written remain available.

The affected worker exits and releases its reservation; the scheduler can
allocate the next trial in a fresh worker while the attempt budget remains.
An OOM-pruned trial consumes one of the 400 attempts and does not satisfy the
250 finite-COMPLETE target. The engine does not silently change the failed
trial's model or batch settings. A killed process or unexplained worker crash
without explicit OOM evidence remains an error.

The three finalists still receive all nine fresh-seed confirmation runs without
performance pruning. Their original early stopping remains enabled. Set
**PRUNING = None** to disable performance pruning in a new study; changing the
policy requires a fresh results directory because it affects candidate selection.
Confirmations are separate runs outside Optuna search-trial pruning; an OOM
during confirmation remains a failed confirmation that needs attention.

See the [Optuna PercentilePruner documentation](https://optuna.readthedocs.io/en/v4.9.0/reference/generated/optuna.pruners.PercentilePruner.html).

## Live TensorBoard

This uses the same TensorBoard notebook extension as
`13_CIFAR10_Joint_HPO.ipynb`. Open it before starting a search stage so metrics
remain visible while its workers run. The dashboard includes the complete study
and confirmation tree beneath **TENSORBOARD_DIRECTORY**.

The existing `common.train.train_model` TensorBoard callback writes epoch metrics
and hyperparameter text. Fresh and resumed search trials now share
`study_root/tensorboard`; the existing HPO outcome writer adds final validation
objectives, trial status, parameters, duration and available divergence evidence.
Resource-pruned search trials also retain the OOM reason and exception
diagnostics through this shared outcome writer.
Each confirmation attempt writes its own epoch metrics and hyperparameter text
under `notebook_runner/confirmations/.../tensorboard`; final confirmation scores
remain in the comparison table and saved receipts.

No custom training callback or duplicate HPO event writer is defined in this
notebook. The dashboard starts empty until trials create events. Hosted runtime
support controls how its embedded view or port 6006 is exposed.

In [ ]:
from IPython import get_ipython


TENSORBOARD_DIRECTORY.mkdir(parents=True, exist_ok=True)
get_ipython().run_line_magic("load_ext", "tensorboard")
get_ipython().run_line_magic(
    "tensorboard", 
    f'--logdir "{TENSORBOARD_DIRECTORY}" --port 6006 --bind_all'
)

## Search space

The engine uses the maintained common.hpo suggestions; this table documents the
default space rather than reimplementing it.

| Group | Parameters and choices |
| --- | --- |
| Capacity | dim 32/64/96/128, always 4 attention heads |
| Patches | size 2/4 when divisible; CNN patchification off/on |
| Backbone | plain or compact U-DiT when patch grid permits |
| Plain depth | 2/3/4/5/6 |
| U-DiT | fixed nine stages; resampling positions 2d_sincos/new_weight |
| Transformer | MLP ratio 2/4; drop-path 0/0.05/0.1; refiner CNN off/on |
| Optimization | Adam/AdamW; learning rate 3e-4 to 5e-3, log-uniform |
| AdamW only | weight decay 1e-6 to 1e-3, log-uniform |
| Batching | 32/64/128 |
| Clipping | per-variable None/0.5/1/5; global None/0.5/1/5 only if per-variable is None |
| Diffusion | 500/1000 timesteps; clipped_cosine/squaredcos_cap_v2/linear |
| Conditioning | p_uncond 0.05/0.1/0.2 |
| EMA | decay 0.995/0.999 |
| Auxiliary image loss | coefficient 0/0.01/0.05/0.1 |

MSE, ordinary-fit cosine learning-rate decay, EMA evaluation, CFG support and
adaptive normalization are fixed. Wrapper sampling uses 50 steps, CFG 4, eta 0;
final image reports use CFG 3. These sampling settings are not optimized.
This study uses no distillation, reconstruction-mode override, or continual replay.

## HPO engine, TensorBoard and recovery

Each stage calls **run_search**, which invokes the existing **common.hpo.run_hpo**
API with the printed settings, including `concurrent_trials`, `worker_gpu_ids`, `pruning`, and the verified
`worker_gpu_memory_limit_mb`. Its shared process scheduler allocates trials,
saves YAML configurations, launches **common.hpo_worker**, and records outcomes.
The runner supplies GPU admission through the API's worker resource context.
With one worker, the same API runs serially in an admitted child.

Each search batch allocates at most **BATCH_TRIALS** new trials (at least
**CONCURRENT_TRIALS** by default), with at most the requested
number training together. The shared API's total-budget mode bounds attempts.
The notebook counts finite COMPLETE trials separately and requests another bounded
batch when OOM pruning, numerical divergence or performance pruning leaves
the target incomplete. OOM-pruned workers release their reservations before
the scheduler starts replacement workers; unclassified crashes still surface.

The SQLite study, sampler state, study specification, YAML configurations,
training histories and reports persist. Re-executing a reached stage adds no
trials. Recovery uses the engine's existing study lock and checkpoint protocol.
Changing `GPU_IDS` or `CONCURRENT_TRIALS` with the same source version preserves
the scientific recipe;
parallel completion order can still change adaptive suggestions.

Use a **new RESULTS_PATH** after changing the dataset, training budget, startup
budget, seed, pruning policy, source implementation or validation protocol. This revision defaults
to **dit_generation_hpo_v6** because the shared engine now classifies recognized
memory exhaustion as resource pruning and its training/source identity changed.
Keep any earlier experiment directory intact. Trial targets may increase without
changing the scientific recipe. No official test-set metric is used for selection.

In [ ]:
display(search_summary(plan))

In [ ]:
first_stage = run_search(
    plan, 
    target_completed=REVIEW_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not first_stage["target_reached"]:
    raise RuntimeError("Attempt ceiling reached before the review target; inspect failed-trial logs.")

## Review after 100 successful trials

Inspect failures, coverage of plain/U-DiT, the best validation loss, and trial
durations. Do not remove a branch only because a few random trials performed
poorly. The next stage keeps the same data split and training/evaluation budget.

In [ ]:
import optuna


study = optuna.load_study(
    study_name=plan["study_name"], 
    storage="sqlite:///" + (Path(plan["study_root"]) / "study.db").as_posix()
)
trial_table = study.trials_dataframe()
display(trial_table.sort_values("value").head(15))
display(search_summary(plan))

In [ ]:
main_stage = run_search(
    plan, 
    target_completed=SEARCH_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not main_stage["target_reached"]:
    raise RuntimeError("Attempt ceiling reached before the main target; inspect the search status.")

## Optional extension

Review the best-loss improvement over the last 50 successful trials and branch
coverage. These summaries are descriptive; they do not prove convergence.
EXTENSION_TARGET=250 adds 50 successful trials after the main search, reaching 250.
Set 200 to skip the extension or 300 to add 100, **before freezing finalists**.
Fresh-seed confirmation provides a separate check on ranking stability.

In [ ]:
if EXTENSION_TARGET not in (200, 250, 300) or EXTENSION_TARGET < SEARCH_TARGET:
    raise ValueError("The extension must preserve the search budget and be 200, 250 or 300.")
final_stage = run_search(
    plan, 
    target_completed=EXTENSION_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
if not final_stage["target_reached"]:
    raise RuntimeError("Resolve the incomplete search target before selecting finalists.")
display(final_stage)

## Freeze and confirm three finalists

The next cell freezes the three best distinct configurations, copies their
**original input configurations**, and records hashes and the paired seed list.
After selection is frozen this runner will not extend the search.

Confirmation uses public load_config/get_datasets/get_model/train_model/report
APIs. It first constructs the original train/validation split and shuffle stream
using SEARCH_SEED, then initializes a new model with each fresh training seed.
It never restores the candidate's trained weights. The sampled architecture,
optimizer, losses, epoch limit and early-stopping rule remain unchanged.
Search pruning and its worker communication are removed from each confirmation.
These reruns are outside Optuna search-trial pruning: confirmation OOMs remain
failed attempts and must be resolved before selecting from complete seed means.

The same three seeds are used for every finalist. Training and final stochastic
evaluation vary by confirmation seed; this is not a decomposition of individual
noise sources. Completed repeats are authenticated and skipped on rerun.
Interrupted attempts are retained and a fresh attempt directory is used.

In [ ]:
finalists = freeze_finalists(plan, CONFIRMATION_SEEDS, top_k=TOP_K)
display(finalists)

In [ ]:
confirmation_records = run_confirmations(plan)
print("Completed confirmation runs:", len(confirmation_records))

In [ ]:
import pandas as pd


summary = pd.DataFrame(confirmation_summary(plan))
display(summary.sort_values("mean_noise_loss"))
if not summary["all_seeds_complete"].all():
    raise RuntimeError("Some confirmation repeats are incomplete; do not select from partial means.")
summary.to_csv(Path(plan["control_root"]) / "confirmation_summary.csv", index=False)

## Interpretation and saved artifacts

Compare mean validation noise_loss and sample standard deviation across the
three paired seeds. Inspect individual receipts and histories when candidate
differences are small. Three repeats are a practical stability check, not proof
that one configuration is universally better.

The study optimizes a denoising proxy. Image quality, diversity and downstream
replay usefulness require separate evaluation after configuration selection.
Keep the official test set untouched until the final evaluation plan is fixed.

The study directory contains study.db, trials.csv, configs and runs.
notebook_runner contains recipe.json, search_status.json, worker requests/logs,
finalists.json, frozen YAML inputs, confirmation receipts and summary tables.
Partial/failed artifacts are retained. All notebook outputs are initially empty.